In [1]:
from pathlib import Path
import pandas as pd
import argparse
import json

In [3]:
# Lists of relevant ICD9 and 10 codes for immunosuppressed patients. Taken from supplementary material of reference study.
ICD9_IMMUNO = []
ICD10_IMMUNO = []

In [4]:
# Loads tables as DataFrames
print('> Loading tables...', end='\r')
df_icu = pd.read_csv(PATH_MIMICIV / 'icustays.csv.gz')
df_patients = pd.read_csv(PATH_MIMICIV / 'patients.csv.gz')
df_icd = pd.read_csv(PATH_MIMICIV / 'diagnoses_icd.csv.gz')
df_dicd = pd.read_csv(PATH_MIMICIV / 'd_icd_diagnoses.csv.gz')

In [5]:
# Load ICD codes
with open(FPATH_JSON_IMMUNO) as f:
    immuno_icds = json.load(f)

    ICD9_IMMUNO = immuno_icds['icd9']
    ICD10_IMMUNO = immuno_icds['icd10']

In [6]:
# Selects relevant entries of immunosuppressed ICD codes
# Treats ICD9 and 10 separately to avoid collisions
df_icd9 = df_icd.loc[df_icd.icd_version == 9]
df_icd10 = df_icd.loc[df_icd.icd_version == 10]

In [7]:
# Fetch HADM_IDs consistent with immunosuppressed ICDs

# Ensures correct datatype prior to comparison

df_hadmid_immuno = pd.concat([
    df_icd9.loc[df_icd9.astype({'icd_code': 'str'}).icd_code.isin(ICD9_IMMUNO)].hadm_id,
    df_icd10.loc[df_icd10.astype({'icd_code': 'str'}).icd_code.isin(ICD10_IMMUNO)].hadm_id
])

# Gets unique IDs
hadmid_immuno = df_hadmid_immuno.unique()

len(hadmid_immuno)

85969

In [8]:
hadmid_immuno

array([28979390, 27897940, 25849114, ..., 29355057, 29889147, 29956342])

In [9]:
df_icu

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
2,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
3,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113
4,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588
...,...,...,...,...,...,...,...,...
73176,19999442,26785317,32336619,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2148-11-19 14:23:43,2148-11-26 13:12:15,6.950370
73177,19999625,25304202,31070865,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-10 19:18:00,2139-10-11 18:21:28,0.960741
73178,19999828,25744818,36075953,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2149-01-08 18:12:00,2149-01-10 13:11:02,1.790995
73179,19999840,21033226,38978960,Trauma SICU (TSICU),Surgical Intensive Care Unit (SICU),2164-09-12 09:26:28,2164-09-17 16:35:15,5.297766


In [10]:
## Drop non-first ICU admissions

# Selects entries related to ICU stays and keeps only the first stay of each admission
df_icu_immuno = df_icu.loc[df_icu.hadm_id.isin(hadmid_immuno)]

print(len(df_icu_immuno))

df_icu_immuno['intime'] = pd.to_datetime(df_icu_immuno['intime'], format='%Y-%m-%d %H:%M:%S')
df_icu_immuno = df_icu_immuno.loc[df_icu_immuno.groupby(by=['hadm_id']).intime.idxmin()]

print(len(df_icu_immuno))

17460
15467


/tmp/ipykernel_10206/1620176952.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icu_immuno['intime'] = pd.to_datetime(df_icu_immuno['intime'], format='%Y-%m-%d %H:%M:%S')


In [11]:
# Drops patients younger than 18 yo

df_icu_immuno = df_icu_immuno.join(df_patients.set_index('subject_id')[['anchor_age']], on='subject_id', how='inner')
df_icu_immuno = df_icu_immuno[df_icu_immuno.anchor_age >= 18]

# Note: Alternatively, get obfuscated birth year from anchor_year - anchor_age, then compare the result against the year in intime
#   to get the patient's age at that specific admission. Doing it with age instead because there's no way of knowing the full birth day.

print(len(df_icu_immuno))

15467


In [12]:
# Drops ICU stays shorter than 6h

df_icu_immuno['outtime'] = pd.to_datetime(df_icu_immuno['outtime'], format='%Y-%m-%d %H:%M:%S')
df_icu_immuno['stay_length'] = df_icu_immuno['outtime'] - df_icu_immuno['intime']
df_icu_immuno = df_icu_immuno.loc[df_icu_immuno.stay_length.dt.total_seconds() >= (6*60*60)]

print(len(df_icu_immuno))

15301


In [13]:
# Exports json with list of qualifying ICU stays
opath = FPATH_JSON_IMMUNO.parent / f'stays_tmp_{FPATH_JSON_IMMUNO.name}'
with open(opath, 'w') as f:
    json.dump(df_icu_immuno.stay_id.to_list(), f)

In [14]:
df_icu_immuno

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,anchor_age,stay_length
44050,16003661,20001305,36916968,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2178-03-25 02:59:09,2178-03-27 21:46:10,2.782650,76,2 days 18:47:01
43849,15975141,20001494,36865385,Neuro Intermediate,Neuro Intermediate,2125-10-26 17:34:33,2125-10-28 19:33:26,2.082558,63,2 days 01:58:53
61127,18346781,20002270,35497510,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2156-08-31 14:52:54,2156-09-01 02:07:59,0.468808,46,0 days 11:15:05
18405,12504470,20003425,36907946,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2155-07-22 17:13:18,2155-07-26 17:11:37,3.998831,76,3 days 23:58:19
51283,17002995,20006309,31646901,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2177-12-02 02:36:00,2177-12-06 16:27:04,4.577130,51,4 days 13:51:04
...,...,...,...,...,...,...,...,...,...,...
59725,18164304,29997500,34008495,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2115-05-18 14:02:33,2115-05-21 20:46:21,3.280417,70,3 days 06:43:48
72460,19907191,29997536,36148756,Neuro Stepdown,Neuro Stepdown,2154-06-13 21:11:33,2154-06-24 22:41:48,11.062674,48,11 days 01:30:15
51035,16968104,29998702,37074618,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2190-11-13 22:42:00,2190-11-15 08:21:09,1.402187,59,1 days 09:39:09
50071,16832788,29998928,34390670,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2115-06-11 19:17:54,2115-06-12 20:40:09,1.057118,78,1 days 01:22:15
